# Phase 1 — Conditional WGAN-GP for Earthquake Spectral Acceleration (SA)

This notebook trains a **Conditional Wasserstein GAN with Gradient Penalty** to learn
the mapping from physical metadata (*Mw, Rrup, Ztor, Vs30*), a tectonic-category
one-hot vector (*interplate / intraplate*), and spectral period (*T*)
to **base-10 log Spectral Acceleration (log10 SA)**.

A **physics-informed monotonic distance-attenuation** penalty is added to the Generator
loss so that predicted SA decreases with increasing rupture distance.


## 0 · Environment & Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

# ---------- Colab vs. Local detection ----------
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/WGAN_IITM/"
except ModuleNotFoundError:
    IN_COLAB = False
    WORK_DIR = os.getcwd()  # assumed to be WGAN_IITM/

os.chdir(WORK_DIR)
print(f"Working directory → {os.getcwd()}")
print(f"Running on {'Google Colab' if IN_COLAB else 'local machine'}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device → {device}")

# ---------- Reproducibility ----------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1 · Data Loading & Preprocessing

In [ ]:
# ── 1a. Load the raw Excel file ──────────────────────────────────────────────
raw = pd.read_excel("nga_subduction.xlsx")
print(f"Raw shape: {raw.shape}")

# Row 0 in this file is a spurious sub-header row (all NaN for numeric fields).
# Drop it and reset the index.
raw = raw.drop(index=0).reset_index(drop=True)
print(f"After dropping header row: {raw.shape}")
raw.head()


In [ ]:
# ── 1b. Identify metadata, tectonic-category, & spectral-period columns ──────
#
# The actual column names in the file have minor quirks:
#   'Vs30 ' (trailing space), 'ztor' (lowercase).
# We normalise them here to the canonical names Mw, Rrup, Ztor, Vs30.

# Build a rename map for the metadata columns we need
rename_map = {}
for c in raw.columns:
    if isinstance(c, str):
        if c.strip().lower() == "vs30":
            rename_map[c] = "Vs30"
        elif c.strip().lower() == "ztor":
            rename_map[c] = "Ztor"
raw.rename(columns=rename_map, inplace=True)

META_COLS = ["Mw", "Rrup", "Ztor", "Vs30"]
TECTONIC_COL = "Inter_intra_flag"
print("Metadata columns present:", all(c in raw.columns for c in META_COLS))
print("Tectonic column present:", TECTONIC_COL in raw.columns)

# Spectral period columns are the numeric (int/float) headers.
PERIOD_COLS = [c for c in raw.columns if isinstance(c, (int, float))]
print(f"Number of spectral-period columns: {len(PERIOD_COLS)}")
print(f"Periods: {PERIOD_COLS}")


In [ ]:
# ── 1c. Keep only useful columns & drop rows with NaN in metadata/SA ─────────
keep_cols = META_COLS + [TECTONIC_COL] + PERIOD_COLS
df = raw[keep_cols].copy()
df[META_COLS] = df[META_COLS].astype(float)
df[TECTONIC_COL] = df[TECTONIC_COL].astype(int)
df[PERIOD_COLS] = df[PERIOD_COLS].astype(float)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Clean wide-format shape: {df.shape}")


In [ ]:
# ── 1d. Melt to long format ──────────────────────────────────────────────────
# Each row becomes (Mw, Rrup, Ztor, Vs30, Inter_intra_flag, Period, SA).
df_long = df.melt(
    id_vars=META_COLS + [TECTONIC_COL],
    value_vars=PERIOD_COLS,
    var_name="Period",
    value_name="SA",
)
df_long["Period"] = df_long["Period"].astype(float)
df_long["SA"] = df_long["SA"].astype(float)
print(f"Long-format shape: {df_long.shape}")
df_long.head()


In [ ]:
# ── 1e. Tectonic one-hot encoding + log transforms ───────────────────────────
# Inter_intra_flag is interpreted as:
#   0 -> interplate / interface
#   1 -> intraplate / intraslab
# If your source file uses the opposite convention, swap the assignments below.
df_long["is_interplate"] = (df_long[TECTONIC_COL] == 0).astype(float)
df_long["is_intraplate"] = (df_long[TECTONIC_COL] == 1).astype(float)
df_long["event_type"] = np.where(
    df_long["is_interplate"] == 1.0,
    "Interplate",
    "Intraplate",
)

# Period = 0 corresponds to PGA (T → 0).  We replace it with a small value
# (1e-3 s ≈ 0.001 s) so that log10(Period) is finite.
PGA_REPLACEMENT = 1e-3
df_long["Period"] = df_long["Period"].replace(0.0, PGA_REPLACEMENT)

df_long["log_Rrup"] = np.log(df_long["Rrup"])
df_long["log_Vs30"] = np.log(df_long["Vs30"])
df_long["log10_Period"] = np.log10(df_long["Period"])
df_long["log10_SA"] = np.log10(df_long["SA"])

print("One-hot tectonic counts:")
print(df_long[["is_interplate", "is_intraplate"]].drop_duplicates().reset_index(drop=True))
print("\nLog-transform summary:")
df_long[["log_Rrup", "log_Vs30", "log10_Period", "log10_SA"]].describe().round(3)


In [ ]:
# ── 1f. Standard-scale the continuous conditioning features ──────────────────
# We scale: Mw, log_Rrup, Ztor, log_Vs30, log10_Period.
# The tectonic one-hot columns are kept as 0/1 inputs.
# log10_SA (the target) is *not* scaled — the Generator learns it directly.

CONTINUOUS_CONDITION_COLS = ["Mw", "log_Rrup", "Ztor", "log_Vs30", "log10_Period"]
TECTONIC_ONEHOT_COLS = ["is_interplate", "is_intraplate"]
META_FEATURE_COLS = ["Mw", "log_Rrup", "Ztor", "log_Vs30"] + TECTONIC_ONEHOT_COLS

scaler = StandardScaler()
df_long[CONTINUOUS_CONDITION_COLS] = scaler.fit_transform(df_long[CONTINUOUS_CONDITION_COLS])

# Save the scaler for downstream reuse
joblib.dump(scaler, "condition_scaler.pkl")
print("Scaler saved → condition_scaler.pkl")

df_long[CONTINUOUS_CONDITION_COLS + TECTONIC_ONEHOT_COLS + ["log10_SA"]].describe().round(3)


In [ ]:
# ── 1g. Train / Test Split & PyTorch Datasets ────────────────────────────────
from sklearn.model_selection import train_test_split

# 80/20 split
df_train, df_test = train_test_split(df_long, test_size=0.20, random_state=SEED)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train set: {len(df_train):,} samples")
print(f"Test  set: {len(df_test):,} samples")


class SADataset(Dataset):
    """Each sample: (scaled continuous + one-hot metadata, scaled log10 period, log10 SA)."""

    def __init__(self, dataframe):
        self.meta = torch.tensor(
            dataframe[META_FEATURE_COLS].values,
            dtype=torch.float32,
        )
        self.period = torch.tensor(
            dataframe["log10_Period"].values, dtype=torch.float32
        ).unsqueeze(1)
        self.sa = torch.tensor(
            dataframe["log10_SA"].values, dtype=torch.float32
        ).unsqueeze(1)
        # Store unscaled Period for plotting (needed for residuals vs period)
        self.period_raw = torch.tensor(
            dataframe["Period"].values, dtype=torch.float32
        ).unsqueeze(1)

    def __len__(self):
        return len(self.sa)

    def __getitem__(self, idx):
        return self.meta[idx], self.period[idx], self.sa[idx]


BATCH_SIZE = 256 * 8

train_dataset = SADataset(df_train)
test_dataset = SADataset(df_test)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

print(f"Train batches/epoch: {len(train_dataloader):,}")
print(f"Test  batches      : {len(test_dataloader):,}")

# Quick shape sanity check
meta_b, per_b, sa_b = next(iter(train_dataloader))
print(f"Batch shapes → meta {meta_b.shape}, period {per_b.shape}, SA {sa_b.shape}")


## 2 · Model Architecture

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
NOISE_DIM = 32         # dimension of latent noise vector z
META_DIM = 6           # Mw, log_Rrup, Ztor, log_Vs30, interplate, intraplate
PERIOD_EMB_DIM = 16    # output dim of the period embedding MLP
HIDDEN_DIM = 128       # width of hidden layers
LAMBDA_GP = 10         # gradient-penalty coefficient
LAMBDA_MONO = 10       # physics monotonicity penalty coefficient
N_CRITIC = 5           # critic updates per generator update
LR = 1e-4
BETAS = (0.5, 0.9)
NUM_EPOCHS = 100


In [ ]:
# ── Period Embedding MLP ─────────────────────────────────────────────────────
# Maps the 1-d scaled log10(Period) to a richer PERIOD_EMB_DIM-d representation
# that both Generator and Critic share.

class PeriodEmbedding(nn.Module):
    def __init__(self, emb_dim=PERIOD_EMB_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, emb_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(emb_dim, emb_dim),
            nn.LeakyReLU(0.2),
        )

    def forward(self, period):          # period: (B, 1)
        return self.net(period)         # → (B, emb_dim)


In [ ]:
# ── Residual Block ───────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    """Pre-activation residual block with LayerNorm."""

    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.LeakyReLU(0.2),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.LeakyReLU(0.2),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        return x + self.block(x)

In [ ]:
# ── Generator ─────────────────────────────────────────────────────────────────
# Input : [z (NOISE_DIM) | metadata (META_DIM) | period_emb (PERIOD_EMB_DIM)]
# Output: single scalar — predicted log10(SA)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.period_emb = PeriodEmbedding(PERIOD_EMB_DIM)
        in_dim = NOISE_DIM + META_DIM + PERIOD_EMB_DIM

        self.net = nn.Sequential(
            nn.Linear(in_dim, HIDDEN_DIM),
            nn.LeakyReLU(0.2),
            ResBlock(HIDDEN_DIM),
            ResBlock(HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.LeakyReLU(0.2),
            nn.Linear(HIDDEN_DIM, 1),      # → predicted log10(SA), no sigmoid
        )

    def forward(self, z, meta, period):
        """z: (B, NOISE_DIM), meta: (B, 6), period: (B, 1)"""
        p_emb = self.period_emb(period)    # (B, PERIOD_EMB_DIM)
        x = torch.cat([z, meta, p_emb], dim=1)
        return self.net(x)                 # (B, 1)


In [ ]:
# ── Critic (Discriminator) ────────────────────────────────────────────────────
# Input : [metadata (META_DIM) | period_emb (PERIOD_EMB_DIM) | SA_value (1)]
# Output: single critic score (unbounded)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.period_emb = PeriodEmbedding(PERIOD_EMB_DIM)
        in_dim = META_DIM + PERIOD_EMB_DIM + 1   # +1 for SA value

        self.net = nn.Sequential(
            nn.Linear(in_dim, HIDDEN_DIM),
            nn.LeakyReLU(0.2),
            ResBlock(HIDDEN_DIM),
            ResBlock(HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.LeakyReLU(0.2),
            nn.Linear(HIDDEN_DIM, 1),
        )

    def forward(self, meta, period, sa):
        """meta: (B, 6), period: (B, 1), sa: (B, 1) — real or fake."""
        p_emb = self.period_emb(period)
        x = torch.cat([meta, p_emb, sa], dim=1)
        return self.net(x)                 # (B, 1)


In [ ]:
# Instantiate & move to device
G = Generator().to(device)
D = Critic().to(device)

opt_G = optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_D = optim.Adam(D.parameters(), lr=LR, betas=BETAS)

print(f"Generator  params: {sum(p.numel() for p in G.parameters()):,}")
print(f"Critic     params: {sum(p.numel() for p in D.parameters()):,}")

## 3 · Loss Functions

In [ ]:
# ── 3a. Gradient Penalty (WGAN-GP) ────────────────────────────────────────────

def gradient_penalty(critic, meta, period, real_sa, fake_sa):
    """Compute the gradient penalty on interpolated samples."""
    B = real_sa.size(0)
    alpha = torch.rand(B, 1, device=device)
    interpolated = (alpha * real_sa + (1 - alpha) * fake_sa).requires_grad_(True)

    d_interp = critic(meta, period, interpolated)

    grads = torch.autograd.grad(
        outputs=d_interp,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
    )[0]

    # grads shape: (B, 1)
    grad_norm = grads.norm(2, dim=1)
    gp = ((grad_norm - 1.0) ** 2).mean()
    return gp

In [ ]:
# ── 3b. Physics-Informed Monotonic Distance-Attenuation Penalty ───────────────
#
# Physical prior: for the SAME earthquake (same Mw, Ztor, Vs30, event type, Period),
# if  R1 < R2  then  SA(R1) >= SA(R2)  ⟹  log10_SA(R1) >= log10_SA(R2).
#
# Implementation:
#   • Within each batch we perturb only the Rrup feature.
#   • We create a "close" and "far" version by shifting scaled log_Rrup.
#   • We penalise max(0, SA_far − SA_close) — i.e. any increase with distance.

def monotonic_distance_penalty(generator, meta, period, z):
    """
    Penalise the Generator whenever a larger Rrup yields a higher SA.

    Uses the same batch metadata but with two different Rrup values.
    Column index 1 in `meta` is the scaled log_Rrup.
    """
    B = meta.size(0)

    # Randomly perturb Rrup: create a "close" copy (smaller Rrup) and a "far" copy (larger Rrup)
    delta = torch.abs(torch.randn(B, 1, device=device)) * 0.5  # always positive shift in standardised space

    meta_close = meta.clone()
    meta_far   = meta.clone()
    meta_close[:, 1:2] = meta[:, 1:2] - delta   # smaller log_Rrup → closer
    meta_far[:, 1:2]   = meta[:, 1:2] + delta   # larger  log_Rrup → farther

    # Same noise and period for fair comparison
    sa_close = generator(z, meta_close, period)
    sa_far   = generator(z, meta_far,   period)

    # Penalty: SA should not *increase* with distance
    violation = torch.relu(sa_far - sa_close)     # (B, 1)
    return violation.mean()


## 4 · Training Loop

In [ ]:
# ── History containers ────────────────────────────────────────────────────────
hist = {"d_loss": [], "g_loss": [], "w_dist": [], "gp": [], "mono": []}

print(f"Training for {NUM_EPOCHS} epochs  |  Critic:Generator ratio = {N_CRITIC}:1")
print(f"Batches per epoch: {len(train_dataloader)}")
print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    d_loss_epoch = []
    g_loss_epoch = []
    w_dist_epoch = []
    gp_epoch     = []
    mono_epoch   = []

    for batch_idx, (meta, period, real_sa) in enumerate(train_dataloader):
        meta    = meta.to(device)
        period  = period.to(device)
        real_sa = real_sa.to(device)
        B = meta.size(0)

        # =====================================================================
        #  CRITIC UPDATE  (N_CRITIC times)
        # =====================================================================
        for _ in range(N_CRITIC):
            z = torch.randn(B, NOISE_DIM, device=device)
            with torch.no_grad():
                fake_sa = G(z, meta, period)

            d_real = D(meta, period, real_sa).mean()
            d_fake = D(meta, period, fake_sa).mean()

            gp = gradient_penalty(D, meta, period, real_sa, fake_sa)

            loss_D = d_fake - d_real + LAMBDA_GP * gp

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

        # =====================================================================
        #  GENERATOR UPDATE  (once)
        # =====================================================================
        z = torch.randn(B, NOISE_DIM, device=device)
        fake_sa = G(z, meta, period)
        d_fake_for_G = D(meta, period, fake_sa).mean()

        loss_G_wgan = -d_fake_for_G
        mono_loss = monotonic_distance_penalty(G, meta, period, z)
        loss_G = loss_G_wgan + LAMBDA_MONO * mono_loss

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        # ── Book-keeping ──
        w_dist_epoch.append((d_real - d_fake).item())
        d_loss_epoch.append(loss_D.item())
        g_loss_epoch.append(loss_G.item())
        gp_epoch.append(gp.item())
        mono_epoch.append(mono_loss.item())

    # ── Epoch-level averages ──
    hist["d_loss"].append(np.mean(d_loss_epoch))
    hist["g_loss"].append(np.mean(g_loss_epoch))
    hist["w_dist"].append(np.mean(w_dist_epoch))
    hist["gp"].append(np.mean(gp_epoch))
    hist["mono"].append(np.mean(mono_epoch))

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:3d}/{NUM_EPOCHS} │ "
            f"D_loss {hist['d_loss'][-1]:+.4f} │ "
            f"G_loss {hist['g_loss'][-1]:+.4f} │ "
            f"W_dist {hist['w_dist'][-1]:.4f} │ "
            f"GP {hist['gp'][-1]:.4f} │ "
            f"Mono {hist['mono'][-1]:.4f}"
        )

print("\n✓ Training complete.")

## 5 · Visualization

In [ ]:
# ── 5a. Loss Curves ──────────────────────────────────────────────────────────
epochs = np.arange(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, hist["d_loss"], label="Critic Loss", linewidth=1.2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Critic Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, hist["g_loss"], label="Generator Loss", color="tab:orange", linewidth=1.2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Generator Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, hist["w_dist"], label="Wasserstein Distance", color="tab:green", linewidth=1.2)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("W-distance")
axes[2].set_title("Wasserstein Distance")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → loss_curves.png")

In [ ]:
# ── 5b. Test-Set Evaluation: Real vs. Predicted log10(SA) + RMSE / MAE ───────
from sklearn.metrics import mean_squared_error, mean_absolute_error

G.eval()

# Generate predictions for the ENTIRE test set
meta_test = test_dataset.meta.to(device)
per_test = test_dataset.period.to(device)
real_test = test_dataset.sa.cpu().numpy().flatten()
period_raw_test = test_dataset.period_raw.cpu().numpy().flatten()   # unscaled Period

with torch.no_grad():
    z_test = torch.randn(len(test_dataset), NOISE_DIM, device=device)
    pred_test = G(z_test, meta_test, per_test).cpu().numpy().flatten()

real_test_sa = np.power(10.0, real_test)
pred_test_sa = np.power(10.0, pred_test)

# ── Regression metrics ──
rmse = np.sqrt(mean_squared_error(real_test, pred_test))
mae = mean_absolute_error(real_test, pred_test)
print(f"Test-set RMSE (log10 SA): {rmse:.4f}")
print(f"Test-set MAE  (log10 SA): {mae:.4f}")

# ── Scatter plots in both model space and original SA scale ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

lims_log = [min(real_test.min(), pred_test.min()) - 0.25,
            max(real_test.max(), pred_test.max()) + 0.25]
axes[0].scatter(real_test, pred_test, alpha=0.15, s=8, edgecolors="none")
axes[0].plot(lims_log, lims_log, "r--", linewidth=1.5, label="1:1 line")
axes[0].set_xlim(lims_log)
axes[0].set_ylim(lims_log)
axes[0].set_xlabel("Real log10(SA)")
axes[0].set_ylabel("Predicted log10(SA)")
axes[0].set_title(f"Model Space\nRMSE={rmse:.4f}  MAE={mae:.4f}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect("equal")

lims_sa = [min(real_test_sa.min(), pred_test_sa.min()),
           max(real_test_sa.max(), pred_test_sa.max())]
axes[1].scatter(real_test_sa, pred_test_sa, alpha=0.15, s=8, edgecolors="none")
axes[1].plot(lims_sa, lims_sa, "r--", linewidth=1.5, label="1:1 line")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Real SA")
axes[1].set_ylabel("Predicted SA")
axes[1].set_title("Original SA Scale")
axes[1].legend()
axes[1].grid(True, alpha=0.3, which="both")

fig.suptitle("Real vs. Predicted Spectral Acceleration — Test Set", y=1.02)
plt.tight_layout()
plt.savefig("real_vs_pred.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → real_vs_pred.png")


In [ ]:
# ── 5c. Residuals vs. Spectral Period ────────────────────────────────────────
residuals = real_test - pred_test

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(period_raw_test, residuals, alpha=0.15, s=8, edgecolors="none")
ax.axhline(0, color="r", linestyle="--", linewidth=1.5)
ax.set_xscale("log")
ax.set_xlabel("Spectral Period T (s)  [log scale]")
ax.set_ylabel("Residual  [Real − Predicted log10(SA)]")
ax.set_title("Residuals vs. Spectral Period  —  Test Set")
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("residuals_vs_period.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Residual stats → mean: {residuals.mean():.4f},  std: {residuals.std():.4f}")
print("Saved → residuals_vs_period.png")


In [ ]:
# ── 5d. Response Spectra — Single Earthquake Event from Test Set ─────────────
# Pick one random earthquake event (unique Mw, Rrup, Ztor, Vs30, tectonic type)
# and compare Real vs. Predicted SA across all spectral periods.

event_keys = ["Mw", "log_Rrup", "Ztor", "log_Vs30", "is_interplate", "is_intraplate"]
events = df_test.groupby(event_keys).ngroups
print(f"Unique earthquake events in test set: {events}")

# Randomly select one event
event_groups = df_test.groupby(event_keys)
event_name = list(event_groups.groups.keys())[np.random.randint(events)]
event_df = event_groups.get_group(event_name).sort_values("Period")

# Get the rows' indices into test_dataset
event_idx = event_df.index.values

# Retrieve tensors for this event
meta_ev = test_dataset.meta[event_idx].to(device)
per_ev = test_dataset.period[event_idx].to(device)
real_ev = test_dataset.sa[event_idx].cpu().numpy().flatten()
period_ev = test_dataset.period_raw[event_idx].cpu().numpy().flatten()

with torch.no_grad():
    z_ev = torch.randn(len(event_idx), NOISE_DIM, device=device)
    pred_ev = G(z_ev, meta_ev, per_ev).cpu().numpy().flatten()

real_ev_sa = np.power(10.0, real_ev)
pred_ev_sa = np.power(10.0, pred_ev)

# Sort by period for clean line plots
sort_idx = np.argsort(period_ev)
period_ev = period_ev[sort_idx]
real_ev_sa = real_ev_sa[sort_idx]
pred_ev_sa = pred_ev_sa[sort_idx]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(period_ev, real_ev_sa, "o-", label="Real SA", markersize=4, linewidth=1.5)
ax.plot(period_ev, pred_ev_sa, "s--", label="Predicted SA", markersize=4, linewidth=1.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Spectral Period T (s)  [log scale]")
ax.set_ylabel("SA")

# Inverse-scale the continuous event metadata for the title
cond_scaled = np.zeros((1, len(CONTINUOUS_CONDITION_COLS)))
cond_scaled[0, 0] = event_name[0]   # Mw (scaled)
cond_scaled[0, 1] = event_name[1]   # log_Rrup (scaled)
cond_scaled[0, 2] = event_name[2]   # Ztor (scaled)
cond_scaled[0, 3] = event_name[3]   # log_Vs30 (scaled)
cond_scaled[0, 4] = 0.0             # dummy log10_Period
cond_orig = scaler.inverse_transform(cond_scaled)[0]
mw_orig = cond_orig[0]
rrup_orig = np.exp(cond_orig[1])
ztor_orig = cond_orig[2]
vs30_orig = np.exp(cond_orig[3])
event_type = event_df["event_type"].iloc[0]

ax.set_title(
    f"Response Spectra — Single Test Event ({event_type})\n"
    f"Mw={mw_orig:.1f}, Rrup={rrup_orig:.0f} km, Ztor={ztor_orig:.1f} km, "
    f"Vs30={vs30_orig:.0f} m/s"
)
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("response_spectra_event.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → response_spectra_event.png")


## 6 · Save Model Weights

In [ ]:
torch.save(G.state_dict(), "global_G.pth")
torch.save(D.state_dict(), "global_D.pth")

print("Saved → global_G.pth")
print("Saved → global_D.pth")
print("Saved → condition_scaler.pkl  (from §1f)")
print("\nPhase 1 complete. These artefacts are ready for Phase 2.")
